# PBMC 1k：单细胞归一化

按照 [sc-best-practices](https://www.sc-best-practices.org/preprocessing_visualization/normalization.html) 实现三种方法。

## 0. 数据和环境

输入为 QC 和 SoupX 后的 `pbmc_1k_qc_filtered.h5ad`，预期大小为 1140 × 12451。scran 需要 R 包 `scran`、`SingleCellExperiment`、`BiocParallel` 和 `Matrix`。

In [ ]:
# Locate the repository without using a machine-specific absolute path.
from pathlib import Path
import shutil
import subprocess
import tempfile

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import io, sparse

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "scripts").exists() or (candidate / "reference").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root")

project_dir = find_project_root()
input_candidates = [
    project_dir / "data/pbmc_1k_qc_filtered.h5ad",
    project_dir / "reference/pbmc_1k_qc_filtered.h5ad",
]
input_path = next((path for path in input_candidates if path.exists()), None)
if input_path is None:
    raise FileNotFoundError(
        "Input file not found. See data/README.md for download instructions."
    )

scran_script = project_dir / "scripts/run_scran_normalization.R"
if not scran_script.exists():
    scran_script = project_dir / "downstream/run_scran_normalization_pbmc_1k.R"

print("Project root:", project_dir)
print("Input:", input_path)
print("scran script:", scran_script)

In [ ]:
# Load the count matrix and preserve raw counts before any transformation.
adata = ad.read_h5ad(input_path)

assert adata.shape == (1140, 12451), ("Unexpected shape:", adata.shape)
assert sparse.issparse(adata.X)
assert np.all(adata.X.data >= 0)
assert np.allclose(adata.X.data, np.round(adata.X.data))

if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()

assert adata.layers["counts"].shape == adata.shape
print(adata)

## 1. 检查原始 library size

In [ ]:
# Library size is the total raw count per cell.
library_size = np.asarray(adata.layers["counts"].sum(axis=1)).ravel()

print("min   =", library_size.min())
print("median=", np.median(library_size))
print("mean  =", library_size.mean())
print("max   =", library_size.max())

## 2. 方法一：Shifted logarithm

`normalize_total(target_sum=None)` 后进行 `log1p`，结果保存到 `log1p_norm`。

In [ ]:
# Normalize each cell to the median raw library size, then apply log1p.
adata_shifted = adata.copy()
adata_shifted.X = adata_shifted.layers["counts"].copy()

scaled = sc.pp.normalize_total(
    adata_shifted,
    target_sum=None,
    inplace=False,
)["X"]

adata.layers["log1p_norm"] = sc.pp.log1p(scaled, copy=True)

In [ ]:
# Verify the target totals and compare raw versus shifted-log distributions.
target_library_size = float(np.median(library_size))
scaled_total = np.asarray(scaled.sum(axis=1)).ravel()

print("Target library size =", target_library_size)
print("Scaled total min    =", scaled_total.min())
print("Scaled total median =", np.median(scaled_total))
print("Scaled total mean   =", scaled_total.mean())
print("Scaled total max    =", scaled_total.max())

assert np.allclose(scaled_total, target_library_size, rtol=1e-6, atol=1e-6)

log1p_total = np.asarray(adata.layers["log1p_norm"].sum(axis=1)).ravel()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(library_size, bins=100)
axes[0].set_title("Raw total counts")
axes[0].set_xlabel("Counts per cell")
axes[1].hist(log1p_total, bins=100)
axes[1].set_title("Shifted logarithm")
axes[1].set_xlabel("Sum of log1p values")
plt.tight_layout()
plt.show()

## 3. 方法二：Scran pooling-based normalization

先生成粗分组，再把 raw counts 交给 scran 计算 size factors。

In [ ]:
# Create coarse groups for scran using a temporary normalized copy.
adata_pp = adata.copy()
adata_pp.X = adata_pp.layers["counts"].copy()

sc.pp.normalize_total(adata_pp)
sc.pp.log1p(adata_pp)
sc.pp.pca(adata_pp, n_comps=15, svd_solver="arpack")
sc.pp.neighbors(adata_pp)
sc.tl.leiden(
    adata_pp,
    key_added="normalization_groups",
    flavor="igraph",
    n_iterations=2,
    directed=False,
)

print(adata_pp.obs["normalization_groups"].value_counts().sort_index())

In [ ]:
# Export sparse counts to R, run scran, and read the size factors back.
counts = adata.layers["counts"].tocsr()
groups = adata_pp.obs["normalization_groups"].astype(str)
assert groups.index.equals(adata.obs_names)

rscript = shutil.which("Rscript")
if rscript is None:
    raise FileNotFoundError("Rscript not found; activate the environment in environment.yml")

probe = subprocess.run(
    [rscript, "-e", "if (!requireNamespace('scran', quietly=TRUE) || !requireNamespace('SingleCellExperiment', quietly=TRUE)) quit(status=1)"],
    capture_output=True,
    text=True,
)
if probe.returncode != 0:
    raise RuntimeError("The selected Rscript does not have scran and SingleCellExperiment installed.")

with tempfile.TemporaryDirectory(prefix="pbmc_1k_scran_") as tmp:
    tmp = Path(tmp)
    counts_path = tmp / "counts_gene_by_cell.mtx"
    groups_path = tmp / "groups.tsv"
    size_factors_path = tmp / "size_factors.tsv"

    io.mmwrite(counts_path, counts.T.tocoo())
    pd.DataFrame({
        "barcode": adata.obs_names.astype(str),
        "group": groups.to_numpy(),
    }).to_csv(groups_path, sep="\t", index=False)

    run = subprocess.run(
        [rscript, str(scran_script), str(counts_path), str(groups_path), str(size_factors_path)],
        capture_output=True,
        text=True,
    )
    if run.stdout:
        print(run.stdout)
    if run.returncode != 0:
        print(run.stderr)
        raise subprocess.CalledProcessError(run.returncode, run.args)
    if run.stderr:
        print("R warnings:\n", run.stderr)

    scran_result = pd.read_csv(size_factors_path, sep="\t")

assert scran_result["barcode"].astype(str).tolist() == adata.obs_names.astype(str).tolist()
size_factors = scran_result["size_factor"].to_numpy()
assert np.all(np.isfinite(size_factors))
assert np.all(size_factors > 0)
adata.obs["scran_size_factors"] = size_factors
print(adata.obs["scran_size_factors"].describe())

### Scran warning

小分组可能出现中间估计 warning。只要最终 size factors 都是 finite 且大于 0，就可以继续，但应记录该 warning。

In [ ]:
# Apply scran size factors to raw counts and log-transform the result.
scran_logged = adata.layers["counts"].multiply(
    1.0 / adata.obs["scran_size_factors"].to_numpy()[:, None]
).tocsr()
scran_logged.data = np.log1p(scran_logged.data)
scran_logged.eliminate_zeros()
adata.layers["scran_normalization"] = scran_logged.astype(np.float32)

## 4. 方法三：Analytic Pearson residuals

直接从 raw counts 计算，结果可以为正或负，不是 TPM 或 log-expression。

In [ ]:
# Calculate analytic Pearson residuals from raw counts without log1p.
adata_pearson = adata.copy()
adata_pearson.X = adata_pearson.layers["counts"].copy()
analytic_pearson = sc.experimental.pp.normalize_pearson_residuals(
    adata_pearson,
    inplace=False,
)["X"]
adata.layers["analytic_pearson_residuals"] = analytic_pearson.astype(np.float32)

## 5. 检查并保存

raw counts 保留在 `.X` 和 `layers["counts"]`，三种方法分别保存。

In [ ]:
# Check shape, layers, raw-count preservation, and finite values.
required_layers = [
    "counts",
    "log1p_norm",
    "scran_normalization",
    "analytic_pearson_residuals",
]
assert adata.shape == (1140, 12451)
assert all(layer in adata.layers for layer in required_layers)
assert np.array_equal(adata.X.toarray(), adata.layers["counts"].toarray())

for layer in required_layers[1:]:
    values = adata.layers[layer]
    values = values.data if sparse.issparse(values) else np.asarray(values)
    assert np.all(np.isfinite(values))

assert np.all(np.isfinite(adata.obs["scran_size_factors"]))
assert np.all(adata.obs["scran_size_factors"] > 0)
print("Normalization integrity check: PASS")

In [ ]:
# Save to results/; do not overwrite the input count matrix.
output_dir = project_dir / "results"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "pbmc_1k_normalized.h5ad"
adata.write_h5ad(output_path)
print("Saved:", output_path)

## 后续使用

- `log1p_norm`：PCA、UMAP 和聚类。
- `scran_normalization`：替代结果和敏感性分析。
- `analytic_pearson_residuals`：高变基因和稀有细胞探索。
- `counts` / `.X`：正式 DE 或 pseudobulk 分析。

三种方法是并列方案，不要串联使用。